# FinSynth Ledger Lab

Block 1: dataset schema registry.


In [ ]:
# Imports
import json
from pathlib import Path
import pandas as pd
from openai import OpenAI
from huggingface_hub import InferenceClient
from dotenv import load_dotenv
import gradio as gr
from generation_runner import generate_raw_json as run_model_generation
from model_options import DEFAULT_MODEL_KEY, get_model_labels


load_dotenv()


In [ ]:
# Create the openAI client for later use
client = OpenAI()
# login(os.environ.get("HF_TOKEN"), add_to_git_credential=True)
hf_client = InferenceClient()

In [ ]:
MAX_TOKENS = 2000  # THIS IS FOR HUGGINGFACE open sourced models, you can define a cap for the output

DATASET_PRESETS = {
    "personal_bank_statement": {
        "label": "Personal bank statement",
        "fields": [
            "date",
            "working",
            "description",
            "withdrawal",
            "deposit",
            "balance",
        ],
        "categories": [
            "salary",
            "groceries",
            "food",
            "transport",
            "subscriptions",
            "transfers",
            "refunds",
        ],
        "rules": [
            # Add your own rules here! to customise each template
            "Return JSON only.",
            "Use fake synthetic financial activity only.",
            "Keep the running balance internally consistent.",
            "Working has only TWO values, Debit or Credit. Data type is string.",
            "You should use the categories only as a guide, generate new categories or even use specific merchant names to make it more realistic.",
        ],
    },
    "credit_card_spending_log": {
        "label": "Credit card spending log",
        "fields": ["date", "description", "amount_spent"],
        "categories": [
            "restaurants",
            "transport",
            "subscriptions",
            "online shopping",
            "groceries",
        ],
        "rules": [
            "Return JSON only.",
            "Use fake synthetic spending data only.",
            "Amounts must be numeric.",
        ],
    },
    "company_expense_ledger": {
        "label": "Company expense ledger",
        "fields": ["date", "description", "amount_spent"],
        "categories": ["SaaS", "office supplies", "travel", "meals", "vendor payments"],
        "rules": [
            "Return JSON only.",
            "Use fake company or vendor style data only.",
            "Descriptions should read like business expenses.",
        ],
    },
}


def get_dataset_preset(dataset_type):
    preset = DATASET_PRESETS.get(dataset_type)
    if preset is None:
        raise ValueError(f"Unknown dataset_type: {dataset_type}")
    return preset


In [ ]:
def build_dataset_prompt(dataset_type, row_count):
    if row_count <= 0:
        raise ValueError("row_count must be positive")

    preset = get_dataset_preset(dataset_type)
    fields = ", ".join(preset["fields"])
    categories = ", ".join(preset["categories"])
    rules = "\n".join(f"- {rule}" for rule in preset["rules"])

    return (
        f"Generate exactly {row_count} rows of synthetic {preset['label'].lower()} data.\n"
        "Return JSON only as a list of objects.\n"
        f"Required fields: {fields}.\n"
        f"Include realistic variety from: {categories}.\n"
        f"Rules:\n{rules}\n"
        "Do not include real personal data, real account numbers, or explanations."
    )


In [ ]:
def validate_dataset_rows(dataset_type, rows, enabled_check=True):
    """Act as a guardrail to further check the json generated after chatGPT api finish polishing the dataset."""

    if not enabled_check:
        return True, "OK"  # just pass the test if the check is disabled.

    preset = get_dataset_preset(dataset_type)
    expected_fields = preset["fields"]
    expected_field_set = set(expected_fields)

    if not isinstance(rows, list):
        return False, "Rows must be a list of objects"

    for index, row in enumerate(rows):
        if not isinstance(row, dict):
            return False, f"Row {index} must be a dictionary"

        actual_field_set = set(row.keys())
        if actual_field_set != expected_field_set:
            return False, f"Row {index} fields must match {expected_fields}"

        if dataset_type == "personal_bank_statement" and row["working"] not in {
            "Debit",
            "Credit",
        }:
            return False, f"Row {index} working must be Debit or Credit"

        if dataset_type != "personal_bank_statement":
            amount = row["amount_spent"]
            if not isinstance(amount, (int, float)):
                return False, f"Row {index} amount_spent must be numeric"

    return True, "OK"


In [ ]:
def json_to_dataframe(clean_json, dataset_type=None, enabled_check=True):
    try:
        rows = json.loads(clean_json)
    except json.JSONDecodeError as exc:
        return None, f"JSON parsing failed: {exc.msg}"

    df = pd.DataFrame(rows)
    if dataset_type is not None:
        is_valid, message = validate_dataset_rows(
            dataset_type, rows, enabled_check=enabled_check
        )
        if not is_valid:
            return None, message

        if enabled_check:
            # if the check is enabled, that means we want to follow our pre-defined schema for the datasets, because we are checking against that. if not, follow whatever schema that the AI generated (because the user may have modified the prompt and hence unchecked the 'enabled check' checkbox)
            expected_fields = get_dataset_preset(dataset_type)["fields"]
        else:
            expected_fields = df.columns.tolist()  # gives a list of columns, like the one as the value of the 'fields' key in our dataset_preset. this will allow us to display all the columns in the gradio UI
        df = df[expected_fields]

    return df, None


def save_csv(df, filename="synthetic_finance_data.csv"):
    if df is None or df.empty:
        return None, "DataFrame is empty"

    output_path = Path(filename).resolve()
    df.to_csv(output_path, index=False)
    return str(output_path), None


## Defining prompts for JSON generation


In [ ]:
# JSON generation now happens in a single pass with model self-check instructions.


In [ ]:
def generate_raw_json(
    dataset_type,
    prompt,
    temperature,
    max_tokens,
    model_key=DEFAULT_MODEL_KEY,
):
    preset = get_dataset_preset(dataset_type)
    return run_model_generation(
        dataset_type=dataset_type,
        prompt=prompt,
        temperature=temperature,
        max_tokens=max_tokens,
        preset=preset,
        model_key=model_key,
        hf_client=hf_client,
        openai_client=client,
    )


In [ ]:
# generate_raw_json(
#     "credit_card_spending_log",
#     build_dataset_prompt("credit_card_spending_log", 5),
#     0.4,
#     300,
# )


In [ ]:
def run_generation_pipeline(
    dataset_type,
    row_count,
    temperature,
    prompt_override=None,
    max_tokens=MAX_TOKENS,
    model_key=DEFAULT_MODEL_KEY,
    raw_generator=generate_raw_json,
    enabled_check=True,
):
    prompt = prompt_override or build_dataset_prompt(dataset_type, row_count)
    clean_json = raw_generator(
        dataset_type,
        prompt,
        temperature,
        max_tokens,
        model_key=model_key,
    )

    df, warning = json_to_dataframe(
        clean_json, dataset_type, enabled_check=enabled_check
    )
    csv_path = None
    if warning is None:
        csv_path, warning = save_csv(df)

    return {
        "prompt": prompt,
        "raw_text": clean_json,
        "clean_json": clean_json,
        "dataframe": df,
        "csv_path": csv_path,
        "warning": warning,
    }


In [ ]:
def get_default_prompt_for_preset(dataset_type, row_count=5):
    return build_dataset_prompt(dataset_type, row_count)


In [ ]:
DATASET_LABELS = {
    "Personal bank statement": "personal_bank_statement",
    "Credit card spending log": "credit_card_spending_log",
    "Company expense ledger": "company_expense_ledger",
}

MODEL_LABELS = get_model_labels()

print(MODEL_LABELS)


def build_gradio_app():
    def update_prompt(dataset_label, row_count):
        dataset_type = DATASET_LABELS[dataset_label]
        return get_default_prompt_for_preset(dataset_type, row_count)

    def generate(
        dataset_label, model_label, row_count, temperature, prompt, enabled_check
    ):
        dataset_type = DATASET_LABELS[dataset_label]
        model_key = MODEL_LABELS[model_label]
        result = run_generation_pipeline(
            dataset_type=dataset_type,
            row_count=row_count,
            temperature=temperature,
            prompt_override=prompt,
            model_key=model_key,
            enabled_check=enabled_check,
        )
        return (
            result["dataframe"],
            result["clean_json"],
            result["csv_path"],
            result["warning"] or "OK",
        )

    with gr.Blocks() as app:
        dataset_type = gr.Dropdown(
            list(DATASET_LABELS.keys()),
            label="Dataset type",
            value="Personal bank statement",
        )
        # replace Dropdown with Radio for radio buttons
        model_type = gr.Dropdown(
            list(MODEL_LABELS.keys()),
            label="Generation model",
            value="Qwen 2.5 72B Instruct",
        )
        row_count = gr.Slider(5, 50, value=5, step=1, label="Row count")
        temperature = gr.Slider(0.2, 1.0, value=0.4, step=0.1, label="Temperature")
        prompt = gr.Textbox(
            label="Prompt",
            lines=8,
            value=get_default_prompt_for_preset("personal_bank_statement", 5),
        )

        # enable json check
        enable_json_check = gr.Checkbox(
            label="Enable JSON checks (UNCHECK if modifying the prompt)",
            value=True,
        )
        generate_button = gr.Button("Generate")
        dataframe = gr.Dataframe(label="DataFrame preview")
        final_json = gr.Textbox(label="Generated JSON", lines=10)
        csv_file = gr.File(label="CSV download")
        status = gr.Textbox(label="Warning/status")

        dataset_type.change(
            update_prompt, inputs=[dataset_type, row_count], outputs=prompt
        )
        row_count.change(
            update_prompt, inputs=[dataset_type, row_count], outputs=prompt
        )
        generate_button.click(
            generate,
            inputs=[
                dataset_type,
                model_type,
                row_count,
                temperature,
                prompt,
                enable_json_check,
            ],
            outputs=[dataframe, final_json, csv_file, status],
        )

    return app


In [ ]:
app = build_gradio_app()
app.launch(inbrowser=True)